# Chess Engine with Pytorch

### Imports

In [1]:
import os 
import torch
import torch.nn as nn
import torch.optim as optim
import torch._dynamo
from torch.utils.data import DataLoader
from chess import pgn
from tqdm import tqdm
import gc
import torch.backends.cudnn as cudnn
from sklearn.model_selection import train_test_split
import chess
import signal
import sys

stop_training = False

def signal_handler(signum, frame):
    global stop_training
    print("\nStopping after current epoch... Press Ctrl+C again to force quit")
    stop_training = True

signal.signal(signal.SIGINT, signal_handler)


<function _signal.default_int_handler(signalnum, frame, /)>

In [2]:
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.0'  # Adjust based on your GPU
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:1500"


print(f"PyTorch version: {torch.__version__}")
# Disable debug APIs for speed
torch.autograd.set_detect_anomaly(False)
torch.autograd.profiler.profile(False)
torch.autograd.profiler.emit_nvtx(False)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch._dynamo.config.suppress_errors = True


PyTorch version: 2.4.1+cu124


In [3]:
def manage_memory():
    if torch.cuda.is_available():
        # Empty cache if memory usage is high
        if torch.cuda.memory_allocated() > 1.5 * 1024**3:  # 1.5GB threshold My GPU has 2GB of memory
            torch.cuda.empty_cache()
            gc.collect()

### Data Preprocessing


#### Load Data

In [4]:
def load_pgn(file_path):
    games = []
    with open(file_path, "r") as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)	
			
    return games


def filter_games(pgn_files, num_files=10):
    """Split verified games into phases while ensuring move legality."""
    filtered_positions = []
    
    for file in pgn_files[:num_files]:
        with open(file) as f:
            while True:
                try:
                    game = chess.pgn.read_game(f)
                    if game is None:
                        break
                        
                    board = chess.Board()
                    moves = list(game.mainline_moves())
                    if len(moves) < 10:  # Skip very short games
                        continue
                        
                    piece_count = 32
                    positions = []
                    
                    # Validate and collect positions
                    for move_idx, move in enumerate(moves):
                        if move not in board.legal_moves:
                            break
                            
                        # Store current position and intended move
                        positions.append((board.copy(), move))
                        
                        # Track material count
                        if board.is_capture(move):
                            piece_count -= 1
                            
                        board.push(move)
                    
                    if not positions:
                        continue
                        
                    # Split into phases
                    total_positions = len(positions)
                    opening_end = min(20, total_positions // 3)
                    middle_end = total_positions - max(10, total_positions // 4)
                    
                    # Add positions with phase weights
                    filtered_positions.extend(positions[:opening_end])  # Opening
                    filtered_positions.extend(positions[opening_end:middle_end])  # Middlegame
                    filtered_positions.extend(positions[middle_end:] * 2)  # Endgame (doubled)
                    
                except Exception as e:
                    print(f"Error processing game: {str(e)}")
                    continue
                    
    print(f"Collected {len(filtered_positions)} valid positions")
    return filtered_positions


In [5]:
from auxiliary_functs import create_input_for_nn, encode_moves

ModuleNotFoundError: No module named 'auxiliary_functs'

# Convert data into tensors

In [6]:
from dataset import ChessDataset
from model import ChessModel

In [ ]:
from ChessVocabulary import ChessVocabulary

# Get list of PGN files
files = [f"../Database/{file}" for file in os.listdir("../Database") if file.endswith(".pgn")]
LIMIT_OF_FILES = min(len(files), 3)  # Start small for 2GB VRAM

# Debug print statements
print(f"Number of files to process: {LIMIT_OF_FILES}")

games = filter_games(files, num_files=LIMIT_OF_FILES)
print(f"Games Parsed: {len(games)}")

# Create vocabulary and process games
vocab = ChessVocabulary()
X, moves = create_input_for_nn(games)
print(f"X shape: {X.shape}, Moves length: {len(moves)}")

if len(X) == 0 or len(moves) == 0:
    raise ValueError("No data was loaded from games")

# Create encoded moves
vocab.load_or_create(moves)
y = vocab.encode_moves(moves)
print(f"y shape: {len(y)}")

In [8]:
# Add memory monitoring
def log_memory_usage(prefix=""):
    if torch.cuda.is_available():
        memory_allocated = torch.cuda.memory_allocated() / 1024**2
        memory_reserved = torch.cuda.memory_reserved() / 1024**2
        print(f"{prefix} Memory: {memory_allocated:.2f}MB allocated, {memory_reserved:.2f}MB reserved")

In [9]:
@torch.no_grad()
def validate(model, val_loader, criterion, device):
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    
    for inputs, labels in val_loader:
        # Match training data types
        inputs = inputs.to(device, dtype=torch.float32, non_blocking=True)
        labels = labels.to(device, dtype=torch.long, non_blocking=True)
        
        with torch.amp.autocast(device_type='cuda', dtype=torch.float32):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
        
        val_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    return val_loss / len(val_loader), 100. * correct / total

In [ ]:
# Optimize CUDA operations
cudnn.benchmark = True

# Create Dataset and DataLoader with optimizations
dataset = ChessDataset(X, y)

# Split data into train/val using scikit-learn
train_data, val_data = train_test_split(dataset, test_size=0.1, random_state=42)

train_loader = DataLoader(
    train_data, 
    batch_size=32, 
    shuffle=True,
    num_workers=2,  
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=False,  
    drop_last=True
)


val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# Check for GPU and optimize
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f'Using device: {device}')

# Free up memory
del X, y, dataset, train_data, val_data

log_memory_usage("After data loading")

In [11]:
import torch.nn.functional as F

def configure_training(model, num_epochs, len_dataloader):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,  # Reduce learning rate
        weight_decay=0.01,
        eps=1e-8   # Increase epsilon for stability
    )
    
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=5e-4,  # Reduce max learning rate
        epochs=num_epochs,
        steps_per_epoch=len_dataloader,
        pct_start=0.3,
        anneal_strategy='cos',
        div_factor=10.0  # More conservative div factor
    )
    
    criterion = nn.CrossEntropyLoss()
    
    return optimizer, scheduler, criterion


In [ ]:
# Initialize fresh or load existing model
if os.path.exists("../model/pytorch/chess_model.pth"):
    print("Loading existing model checkpoint...")
    checkpoint = torch.load("../model/pytorch/chess_model.pth")
    model = ChessModel(num_classes=checkpoint['vocab'].vocab_size).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    vocab = checkpoint['vocab']  # Load vocabulary from checkpoint
    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint.get('best_loss', float('inf'))
    print(f"Resuming from epoch {start_epoch}")
else:
    print("Initializing new model...")
    model = ChessModel(num_classes=vocab.vocab_size).to(device)
    start_epoch = 0
    best_loss = float('inf')

# Training Configuration
optimizer, scheduler, criterion = configure_training(model, num_epochs=25, len_dataloader=len(train_loader))
scaler = torch.cuda.amp.GradScaler()

# Training settings
accumulation_steps = 8  # Gradient accumulation steps


In [13]:
def train_epoch(model, dataloader, optimizer, scaler, scheduler, criterion, device):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(dataloader, desc=f"Training", leave=False)
    
    for i, (inputs, labels) in enumerate(pbar):
        try:
            # Ensure consistent data types
            inputs = inputs.to(device, dtype=torch.float32, non_blocking=True)
            labels = labels.to(device, dtype=torch.long, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            # Use updated autocast syntax
            with torch.amp.autocast(device_type='cuda', dtype=torch.float32):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.step()
            scheduler.step()
            
            running_loss += loss.item()
            
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'lr': f'{optimizer.param_groups[0]["lr"]:.1e}'
            })
            
        except Exception as e:
            print(f"Error in batch {i}: {str(e)}")
            continue
            
    return running_loss / len(dataloader)


In [ ]:
# Initialize wandb for tracking
import wandb
wandb.init(project="chess_engine", resume=True)

total_epochs = 25
for epoch in range(start_epoch, total_epochs):
    # Training and validation
    train_loss = train_epoch(model, train_loader, optimizer, scaler, scheduler, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Print progress
    print(f'Epoch {epoch+1}/{total_epochs}:')
    print(f'Train Loss: {train_loss:.4f}')
    print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    
    # Log to wandb
    wandb.log({
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "learning_rate": optimizer.param_groups[0]['lr']
    })
    
    # Save best model
    if val_loss < best_loss:
        best_loss = val_loss
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss': val_loss,
            'best_loss': best_loss,
            'vocab': vocab,
        }
        torch.save(checkpoint, '../model/pytorch/chess_model.pth')
        print(f"Saved new best model with loss: {val_loss:.4f}")
    
    # Regular checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss': val_loss,
            'best_loss': best_loss,
            'vocab': vocab,
        }
        torch.save(checkpoint, f'../model/pytorch/chess_model_epoch_{epoch+1}.pth')
    
    log_memory_usage(f"Epoch {epoch}")

In [ ]:
# Initialize wandb for tracking
import wandb
wandb.init(project="chess_engine", resume=True)
total_epochs = 25
for epoch in range(start_epoch, total_epochs + start_epoch):
    # Training and validation
    train_loss = train_epoch(model, train_loader, optimizer, scaler, scheduler, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Print progress
    print(f'Epoch {epoch+1}/{total_epochs}:')
    print(f'Train Loss: {train_loss:.4f}')
    print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    
    # Log to wandb
    wandb.log({
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "learning_rate": optimizer.param_groups[0]['lr']
    })
    
    # Save best model
    if val_loss < best_loss:
        best_loss = val_loss
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss': val_loss,
            'best_loss': best_loss,
            'vocab': vocab,
        }
        torch.save(checkpoint, '../model/pytorch/chess_model.pth')
        print(f"Saved new best model with loss: {val_loss:.4f}")
    
    # Regular checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss': val_loss,
            'best_loss': best_loss,
            'vocab': vocab,
        }
        torch.save(checkpoint, f'../model/pytorch/chess_model_epoch_{epoch+1}.pth')
    
    log_memory_usage(f"Epoch {epoch}")

In [ ]:
# Save model and cleanup
torch.save({
	'epoch': total_epochs + start_epoch,
	'model_state_dict': model.state_dict(),
	'optimizer_state_dict': optimizer.state_dict(),
	'loss': val_loss,
}, '../model/pytorch/best_model.pth')

torch.cuda.empty_cache()
gc.collect()


In [17]:
torch.save({
    'epoch': total_epochs + start_epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'vocab': vocab,
}, '../model/pytorch/chess_model.pth')

In [ ]:
def train_continuous(model, optimizer, scaler, criterion, device, files, batch_size=32):
    """Memory-efficient continuous training with proper scheduler reset"""
    global stop_training
    file_idx = 0
    best_loss = float('inf')
    
    try:
        while not stop_training:
            print(f"\nLoading files {file_idx} to {file_idx + 3}")
            
            try:
                current_files = files[file_idx:file_idx + 3]
                if not current_files:
                    file_idx = 0
                    current_files = files[file_idx:file_idx + 3]
                    
                # Load and process games
                games = filter_games(current_files)
                X, moves = create_input_for_nn(games)
                print(f"Processed {len(X)} positions into training data")
                
                # Update vocab only for new moves
                vocab.load_or_create(moves)
                y = vocab.encode_moves(moves)
                
                # Create dataset
                dataset = ChessDataset(X, y)
                train_data, val_data = train_test_split(dataset, test_size=0.1)
                
                train_loader = DataLoader(
                    train_data, 
                    batch_size=batch_size,
                    shuffle=True,
                    num_workers=2,
                    pin_memory=True,
                    prefetch_factor=2
                )
                
                # Reset scheduler for new batch of data
                num_epochs = start_epoch + total_epochs
                scheduler = optim.lr_scheduler.OneCycleLR(
                    optimizer,
                    max_lr=5e-4,
                    epochs=num_epochs,
                    steps_per_epoch=len(train_loader),
                    pct_start=0.3,
                    anneal_strategy='cos'
                )
                
                # Train on current batch
                train_loss = train_epoch(model, train_loader, optimizer, 
                                       scaler, scheduler, criterion, device)
                
                # Save if better
                if train_loss < best_loss:
                    best_loss = train_loss
                    print(f"New best loss: {train_loss:.4f}")
                    torch.save({
                        'file_idx': file_idx,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'vocab': vocab,
                        'best_loss': best_loss,
                        'loss': train_loss,
                    }, '../model/pytorch/chess_model_best.pth')
                
                # Regular checkpoint
                torch.save({
                    'file_idx': file_idx,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'vocab': vocab,
                    'loss': train_loss,
                }, '../model/pytorch/chess_model_continuous.pth')
                
                # Clean up
                del dataset, train_loader, train_data, val_data, X, y, games
                torch.cuda.empty_cache()
                gc.collect()
                
                print(f"Memory: {torch.cuda.memory_allocated()/1024**2:.2f}MB allocated, "
                      f"{torch.cuda.memory_reserved()/1024**2:.2f}MB reserved")
                
                file_idx += 3
                
            except Exception as e:
                print(f"Error processing files: {str(e)}")
                file_idx += 3
                continue
                
    except KeyboardInterrupt:
        print("\nTraining interrupted by user. Saving checkpoint...")
        torch.save({
            'file_idx': file_idx,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'vocab': vocab,
            'best_loss': best_loss,
        }, '../model/pytorch/chess_model_interrupted.pth')

# Use this to start training
files = [f"../Database/{file}" for file in os.listdir("../Database") if file.endswith(".pgn")]
train_continuous(model, optimizer, scaler, criterion, device, files)
